In [13]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [14]:
input_data = "input_files/relationship_extraction_0145.csv"
output_data = "output_files/relationship_extraction_0145.xlsx"

In [ ]:
# ## Open AI API
# os.environ["OPENAI_API_KEY"] = "your open ai api"
from dotenv import load_dotenv

load_dotenv()

## common functions

In [16]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [17]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()
        return content

In [ ]:
"""
    This function is for the relationship extraction task.
"""
def construct_annotation(df, results, task_name, base_data_path, rule_id="0145"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema, presentation, calculation, definition, label, instance, us_gaap, true_answer, segmentation_path, parse_ok, input_col, outputByLLM = [[] for _ in range(12)]

    for res in results:
        folder_name = res.get("folder_name")
        prefix_path = os.path.join(base_data_path, folder_name)
        
        parse = False
        cal, pre, defn, lab, ins, sch, seg_path = [None] * 7
        
        ## special for relationship extraction task (DQC.US.0145)
        head_concept = res.get("head_concept")
        tail_concept = res.get("tail_concept")
        
        flattened_answers = [head_concept, tail_concept]
        answers = [(head_concept, "Inappropriateness", tail_concept)]

        if os.path.exists(prefix_path) and os.path.isdir(prefix_path):
            all_subfolders = [d for d in os.listdir(prefix_path) if os.path.isdir(os.path.join(prefix_path, d))]

            for subfolder_name in all_subfolders:
                full_path = os.path.join(prefix_path, subfolder_name)
                temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch = [None] * 6
                
                file_list = os.listdir(full_path)
                for file_name in file_list:
                    final_path = os.path.join(full_path, file_name)
                    if file_name.endswith("cal.xml"): temp_cal = read_xml(final_path)
                    elif file_name.endswith("pre.xml"): temp_pre = read_xml(final_path)
                    elif file_name.endswith("def.xml"): temp_defn = read_xml(final_path)
                    elif file_name.endswith("lab.xml"): temp_lab = read_xml(final_path)
                    elif file_name.endswith("htm.xml"): temp_ins = read_xml(final_path)
                    elif file_name.endswith("xsd"): sch = read_xml(final_path)
                
                is_appear = True
                ## check the presentation linkbase
                presentation_content = temp_pre or ""
                
                if not all(flattened_answers) or not presentation_content:
                    is_appear = False
                else:
                    for ans in flattened_answers:
                        if not (ans in presentation_content or ans.replace(":", "_") in presentation_content):
                            is_appear = False
                            break
               
                if is_appear:
                    parse = True
                    cal, pre, defn, lab, ins, sch = temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch
                    seg_path = os.path.join(folder_name, subfolder_name)
                    break 
        
        calculation.append(cal); presentation.append(pre); definition.append(defn); label.append(lab); instance.append(ins); schema.append(sch);
        true_answer.append(answers); us_gaap.append(None); parse_ok.append(parse); segmentation_path.append(seg_path); input_col.append(None); outputByLLM.append(res);
     
    new_df["schema"] = schema; new_df["presentation"] = presentation; new_df["calculation"] = calculation; new_df["definition"] = definition;
    new_df["label"] = label; new_df["instance"] = instance; new_df["us_gaap"] = us_gaap; new_df["input"] = input_col;
    new_df["true_answer"] = true_answer; new_df["segmentation_path"] = segmentation_path; new_df["parse_successful"] = parse_ok; new_df["LLM_Output"] = outputByLLM;

    return new_df

## load dqc messages for relationship extraction task (DQC.US.0145)

In [19]:
re_0145_df = pd.read_csv(input_data)

In [20]:
re_0145_df.head()

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0145.9867,"2023-02-08 18:00:58,411 [DQC.US.0145.9867] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,US GAAP 2022,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/913144/...,2023-02-08,2023,546806,https://www.sec.gov/Archives/edgar/data/913144...,6331,10k-rnr-20221231
1,DQC.US.0145.9868,"2023-02-08 18:00:58,411 [DQC.US.0145.9868] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,US GAAP 2022,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/913144/...,2023-02-08,2023,546806,https://www.sec.gov/Archives/edgar/data/913144...,6331,10k-rnr-20221231
2,DQC.US.0145.9868,"2023-02-08 18:00:58,412 [DQC.US.0145.9868] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,US GAAP 2022,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/913144/...,2023-02-08,2023,546806,https://www.sec.gov/Archives/edgar/data/913144...,6331,10k-rnr-20221231
3,DQC.US.0145.9867,"2023-02-14 12:30:29,301 [DQC.US.0145.9867] The...",2023-02-14,ERROR,DQC,145,1620749,PANAMERA HOLDINGS CORPORATION,2023-02-14 12:03:00,0001640334-23-000211,US GAAP 2022,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1620749...,2023-02-14,2023,548226,https://www.sec.gov/Archives/edgar/data/162074...,8742,10q-pnht-20221031
4,DQC.US.0145.9867,"2023-02-14 12:30:29,302 [DQC.US.0145.9867] The...",2023-02-14,ERROR,DQC,145,1620749,PANAMERA HOLDINGS CORPORATION,2023-02-14 12:03:00,0001640334-23-000211,US GAAP 2022,XBRL Document Created with XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1620749...,2023-02-14,2023,548226,https://www.sec.gov/Archives/edgar/data/162074...,8742,10q-pnht-20221031


## LLM judge assisstant

In [ ]:
Client = OpenAI()

In [ ]:
re_0145_system_prompt = """You are an expert in XBRL taxonomies, tasked with parsing DQC validation messages for relationship errors (Rule DQC.US.0145), specifically 'Inappropriate Cash Flow Presentation'.

1. **Head Concept**: Extract the primary concept that is inappropriately presented. This is usually the first concept mentioned in the message.

2. **Tail Concept**: Extract the presentation concept that the head concept is incorrectly a descendant of.

3. **Reasoning**: Provide a short explanation that the head concept should not be presented as a component of the tail concept.

Output your result as a single, structured JSON object in the following format:
```json
{
  "head_concept": "us-gaap:EffectOfExchangeRateOnCashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
  "tail_concept": "us-gaap:NetCashProvidedByUsedInFinancingActivitiesAbstract",
  "reason": "The message indicates the head concept is inappropriately presented as a descendant of the tail concept and should be outside this group."
}
```"""

In [23]:
def get_response(dqc_message):
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": re_0145_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \\n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
res = []
# to run on the full dataset, use the line below.
for _, row in tqdm(re_0145_df.iterrows(), total=re_0145_df.shape[0]):
# to test on only the first 10 examples, comment out the line above and uncomment the line below:
# for _, row in tqdm(re_0145_df.head(10).iterrows(), total=10):
    
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    # DQC 0145 messages do not have a statement name, so we only need the folder_name
    response["folder_name"] = row["folder_name"]
    res.append(response)

100%|██████████| 142/142 [03:48<00:00,  1.61s/it]


In [32]:
res[0]

{'head_concept': 'us-gaap:EffectOfExchangeRateOnCashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents',
 'tail_concept': 'us-gaap:NetCashProvidedByUsedInFinancingActivitiesAbstract',
 'reason': 'The message indicates the head concept is inappropriately presented as a descendant of the tail concept and should be outside this group.',
 'folder_name': '10k-rnr-20221231'}

In [ ]:
DATA_FOLDER_PATH = "./segmentation_data/relationship_extraction"

new_re_0145_df = construct_annotation(re_0145_df, res, 
                                     task_name="relationship_extraction", 
                                     base_data_path=DATA_FOLDER_PATH)

In [30]:
new_re_0145_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0145.9867,"2023-02-08 18:00:58,411 [DQC.US.0145.9867] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:EffectOfExchangeRateOnCashCashEquiva...,10k-rnr-20221231/ConsolidatedStatementsofCashF...,True,{'head_concept': 'us-gaap:EffectOfExchangeRate...
1,DQC.US.0145.9868,"2023-02-08 18:00:58,411 [DQC.US.0145.9868] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:IncreaseDecreaseInDueToAffiliates, I...",10k-rnr-20221231/ScheduleIICondensedFinancialI...,True,{'head_concept': 'us-gaap:IncreaseDecreaseInDu...
2,DQC.US.0145.9868,"2023-02-08 18:00:58,412 [DQC.US.0145.9868] The...",2023-02-08,ERROR,DQC,145,913144,RenaissanceRe Holdings Ltd.,2023-02-08 17:28:00,0000913144-23-000011,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[(us-gaap:ProceedsFromContributionsFromAffilia...,10k-rnr-20221231/ScheduleIICondensedFinancialI...,True,{'head_concept': 'us-gaap:ProceedsFromContribu...
3,DQC.US.0145.9867,"2023-02-14 12:30:29,301 [DQC.US.0145.9867] The...",2023-02-14,ERROR,DQC,145,1620749,PANAMERA HOLDINGS CORPORATION,2023-02-14 12:03:00,0001640334-23-000211,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:InterestPaidNet, Inappropriateness, ...",10q-pnht-20221031/StatementsOfCashFlowsUnaudited,True,"{'head_concept': 'us-gaap:InterestPaidNet', 't..."
4,DQC.US.0145.9867,"2023-02-14 12:30:29,302 [DQC.US.0145.9867] The...",2023-02-14,ERROR,DQC,145,1620749,PANAMERA HOLDINGS CORPORATION,2023-02-14 12:03:00,0001640334-23-000211,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:IncomeTaxesPaid, Inappropriateness, ...",10q-pnht-20221031/StatementsOfCashFlowsUnaudited,True,"{'head_concept': 'us-gaap:IncomeTaxesPaid', 't..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,DQC.US.0145.9868,"2024-11-19 14:46:45,367 [DQC.US.0145.9868] The...",2024-11-19,ERROR,DQC,145,87802,"SCIENTIFIC INDUSTRIES, INC.",2024-11-14 16:14:00,0001654954-24-014460,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[(us-gaap:ProceedsFromIssuanceOfCommonStock, I...",10q-scnd-20240930/CondensedConsolidatedStateme...,True,{'head_concept': 'us-gaap:ProceedsFromIssuance...
138,DQC.US.0145.9868,"2024-11-19 14:46:45,368 [DQC.US.0145.9868] The...",2024-11-19,ERROR,DQC,145,87802,"SCIENTIFIC INDUSTRIES, INC.",2024-11-14 16:14:00,0001654954-24-014460,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8

In [31]:
new_re_0145_df.to_excel(output_data, index=False)